In [ ]:
!pip install rasterio torchmetrics geopandas albumentations imagecodecs fiona

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:2"
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import gc
import math
import glob
import shutil
import random
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import List, Tuple, Union
from skimage import io
from scipy.ndimage import distance_transform_edt as edt
from scipy.ndimage import binary_erosion

import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds

import torch
import torch.nn as nn
import torchmetrics
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

import albumentations as A
from albumentations.pytorch import ToTensorV2
from typing import Callable, List, Tuple, Optional
from tqdm.notebook import tqdm

In [ ]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')

def make_gen(nseed):
    g = torch.Generator()
    g.manual_seed(nseed)
    return g

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
base_dir = '/content/drive/MyDrive/AI4SF_uNet_Fusion'
os.chdir(base_dir)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Composite Loss

In [ ]:
class SobelLoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred_logits: torch.Tensor, target: torch.Tensor):
        if target.dim() == 3:
            target = target.unsqueeze(1)
        target = target.float()

        pred_probs = torch.sigmoid(pred_logits).clamp(self.eps, 1.0 - self.eps)

        g_pred = K.filters.spatial_gradient(pred_probs, mode='sobel', normalized=True)
        g_tgt  = K.filters.spatial_gradient(target,      mode='sobel', normalized=True)

        mag_pred = torch.sqrt((g_pred ** 2).sum(dim=2) + 1e-12)
        mag_tgt  = torch.sqrt((g_tgt  ** 2).sum(dim=2) + 1e-12)

        return F.l1_loss(mag_pred, mag_tgt)

In [ ]:
class BCETverskySobelLoss(nn.Module):
    def __init__(
        self,
        bce_weight: float = 1.0,
        tversky_weight: float = 1.0,
        sobel_weight: float = 0.25,
        tversky_alpha: float = 0.5,
        tversky_beta: float = 0.5,
        smooth: float = 1e-6,
        reduction: str = 'mean'
    ):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction=reduction)
        self.tversky = smp.TverskyLoss(
            mode='binary',
            alpha=tversky_alpha,
            beta=tversky_beta,
            eps=smooth,
            from_logits=True
        )
        self.sobel = SobelLoss(eps=smooth)

        self.bce_w = bce_weight
        self.tversky_w = tversky_weight
        self.sobel_w = sobel_weight  # can ramp this externally

        self.last_losses = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if targets.dim() == 3:
            targets = targets.unsqueeze(1)
        targets = targets.float()

        bce_loss = self.bce(logits, targets)
        tversky_loss = self.tversky(logits, targets)
        sobel_loss = self.sobel(logits, targets)

        total = self.bce_w * bce_loss + self.tversky_w * tversky_loss + self.sobel_w * sobel_loss

        self.last_losses = {
            'total': total.detach(),
            'bce': bce_loss.detach(),
            'tversky': tversky_loss.detach(),
            'sobel': sobel_loss.detach(),
            'weights': {
                'bce_w': torch.tensor(self.bce_w),
                'tversky_w': torch.tensor(self.tversky_w),
                'sobel_w': torch.tensor(self.sobel_w),
            }
        }
        return total

## Data Generator for Mid and Late Level Fusion

In [ ]:
class DataGeneratorDual(Dataset):
    def __init__(self, images1_dir, images2_dir, masks_dir, num_classes, augment=False):
        self.images1_dir = images1_dir
        self.images2_dir = images2_dir
        self.masks_dir = masks_dir
        self.num_classes = num_classes
        self.augment = augment

        self.image_s1_list = sorted(f for f in os.listdir(self.images1_dir) if not f.startswith('.'))
        self.image_s2_list = sorted(f for f in os.listdir(self.images2_dir) if not f.startswith('.'))
        self.mask_list     = sorted(f for f in os.listdir(self.masks_dir)   if not f.startswith('.'))

        self.filenames = self.image_s1_list

        self.extents = []
        for f1, f2 in zip(self.image_s1_list, self.image_s2_list):
            with rasterio.open(os.path.join(self.images1_dir, f1)) as src:
                self.extents.append(src.bounds)
            _ = rasterio.open(os.path.join(self.images2_dir, f2))

        if self.augment:
            self.augmentation = A.Compose([A.HorizontalFlip(p=0.5),
                                           A.VerticalFlip(p=0.5),
                                           A.RandomRotate90(p=0.5),
                                           A.Transpose(p=0.5),
                                           A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=45, border_mode=0, p=0.5),
                                           A.RandomResizedCrop(height=256, width=256, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.5),
                                           ToTensorV2()],
                                          additional_targets={"image2": "image"})
        else:
            self.to_tensor = ToTensorV2()

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_S1_path = os.path.join(self.images1_dir, self.image_s1_list[idx])
        img_S2_path = os.path.join(self.images2_dir, self.image_s2_list[idx])
        mask_path = os.path.join(self.masks_dir, self.mask_list[idx])

        img_S1 = rasterio.open(img_S1_path, 'r')
        img_S2 = rasterio.open(img_S2_path, 'r')
        mask = rasterio.open(mask_path, 'r')

        img_array_S1 = img_S1.read(out_dtype=np.float32)
        img_array_S2 = img_S2.read(out_dtype=np.float32)
        mask_array = mask.read(1, out_dtype=np.float32)

        img_nor_S1 = (img_array_S1 - img_array_S1.min(axis=(1, 2), keepdims=True)) / \
                     (img_array_S1.max(axis=(1, 2), keepdims=True) - img_array_S1.min(axis=(1, 2), keepdims=True) + 1e-8)

        img_nor_S2 = (img_array_S2 - img_array_S2.min(axis=(1, 2), keepdims=True)) / \
                     (img_array_S2.max(axis=(1, 2), keepdims=True) - img_array_S2.min(axis=(1, 2), keepdims=True) + 1e-8)

        mask_array = mask_array / 255.0

        if self.augment:
            stream1_aug = img_nor_S1.transpose(1, 2, 0)
            stream2_aug = img_nor_S2.transpose(1, 2, 0)

            augmented = self.augmentation(image=stream1_aug, image2=stream2_aug, mask=mask_array)

            s1_tensor = augmented['image'].permute(2, 0, 1)
            s2_tensor = augmented['image2'].permute(2, 0, 1)
            mask_tensor = augmented['mask'].unsqueeze(0)
        else:
            s1_tensor = torch.from_numpy(img_nor_S1).float()
            s2_tensor = torch.from_numpy(img_nor_S2).float()
            mask_tensor = torch.from_numpy(mask_array).float().unsqueeze(0)

        extent = (self.extents[idx].left,
                  self.extents[idx].bottom,
                  self.extents[idx].right,
                  self.extents[idx].top)

        return [s1_tensor, s2_tensor], mask_tensor, self.filenames[idx], extent

## Data Generator for Early Level Fusion

In [ ]:
class DataGeneratorSingle(Dataset):
    def __init__(self, images_dir, masks_dir, num_classes, augment=False):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.num_classes = num_classes
        self.augment = augment

        self.image_list = sorted(f for f in os.listdir(self.images_dir) if not f.startswith('.'))
        self.mask_list     = sorted(f for f in os.listdir(self.masks_dir)   if not f.startswith('.'))

        self.filenames = self.image_list

        self.extents = []
        for f1 in self.image_list:
            with rasterio.open(os.path.join(self.images_dir, f1)) as src:
                self.extents.append(src.bounds)

        if self.augment:
            self.augmentation = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Transpose(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=45, border_mode=0, p=0.5),
                A.ChannelShuffle(p=0.2),
                A.RandomResizedCrop(height=256, width=256, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.5),
                ToTensorV2()])
        else:
             self.augmentation = ToTensorV2()

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_list[idx])
        mask_path = os.path.join(self.masks_dir, self.mask_list[idx])

        with rasterio.open(img_path) as img:
            img_array = img.read(out_dtype=np.float32)
        with rasterio.open(mask_path) as mask:
            mask_array = mask.read(1, out_dtype=np.float32)

        img_nor = np.zeros(img_array.shape, dtype=np.float32)
        for k in range(img_array.shape[0]):
          band_max = np.max(img_array[k, :, :])
          band_min = np.min(img_array[k, :, :])
          img_nor[k, :, :] = (img_array[k, :, :] - band_min) / (band_max - band_min + 1e-8)

        img_array = img_nor
        mask_array = mask_array / 255.0

        if img_array.shape[1:] != mask_array.shape:
            raise ValueError(f"Image and mask shapes do not match for {self.image_list[idx]}")

        if self.augment:
            img_aug = img_array.transpose(1, 2, 0)
            augmented = self.augmentation(image=img_aug, mask=mask_array)
            img_tensor = augmented['image'].permute(0, 1, 2)
            mask_tensor = augmented['mask'].unsqueeze(0)
        else:
            img_aug = img_array.transpose(1, 2, 0)
            augmented = self.augmentation(image=img_aug, mask=mask_array)
            img_tensor = augmented['image']
            mask_tensor = augmented['mask'].unsqueeze(0)

        extent = (self.extents[idx].left,
                  self.extents[idx].bottom,
                  self.extents[idx].right,
                  self.extents[idx].top)

        return img_tensor, mask_tensor, self.filenames[idx], extent

## Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0.0, path='/content/drive/MyDrive/AI4SF_uNet_Fusion_R1/weights', loss_name='default', fusion_name='default'):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.path = path
        self.loss_name = loss_name
        self.fusion_name = fusion_name

    def __call__(self, val_loss, model, optimizer=None, scheduler=None, epoch=None):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, scheduler, epoch)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, scheduler, epoch)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, optimizer=None, scheduler=None, epoch=None):
        checkpoint = {"model_state_dict": model.state_dict(),
                      "optimizer_state_dict": optimizer.state_dict() if optimizer else None,
                      "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                      "epoch": epoch,
                      "val_loss": val_loss}

        filename = f'{source_tag}_{self.loss_name}_{self.fusion_name}_Unet.pt'
        checkpoint_path = os.path.join(self.path, filename)
        torch.save(checkpoint, checkpoint_path)

        if self.verbose:
            print(f"🔐 Checkpoint saved to {checkpoint_path} at val_loss = {val_loss:.4f}")

## scSE

In [ ]:
class ChannelSE(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelSE, self).__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.pool(x)
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y)
        return x * y

class SpatialSE(nn.Module):
    def __init__(self, in_channels):
        super(SpatialSE, self).__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.conv(x)
        y = self.sigmoid(y)
        return x * y

class SCSEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SCSEBlock, self).__init__()
        self.cSE = ChannelSE(in_channels, reduction)
        self.sSE = SpatialSE(in_channels)

    def forward(self, x):
        return self.cSE(x) + self.sSE(x)

## Encoder-Decoder Blocks

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_scse=True, reduction=16):
        super(EncoderBlock, self).__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                                  nn.BatchNorm2d(out_channels),
                                  nn.ReLU(inplace=True),
                                  nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
                                  nn.BatchNorm2d(out_channels),
                                  nn.ReLU(inplace=True))

        self.scse = SCSEBlock(out_channels, reduction) if use_scse else nn.Identity()

    def forward(self, x):
        x = self.conv(x)
        return self.scse(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels, use_scse=True, reduction=16):
        super(DecoderBlock, self).__init__()
        self.up = nn.Sequential(nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2, bias=False),
                                nn.BatchNorm2d(out_channels),
                                nn.ReLU(inplace=True))

        self.conv = nn.Sequential(nn.Conv2d(out_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
                                  nn.BatchNorm2d(out_channels),
                                  nn.ReLU(inplace=True),
                                  nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
                                  nn.BatchNorm2d(out_channels),
                                  nn.ReLU(inplace=True))

        self.scse = SCSEBlock(out_channels, reduction) if use_scse else nn.Identity()

    def forward(self, x, skip):
        x = self.up(x)

        diffY = skip.size(2) - x.size(2)
        diffX = skip.size(3) - x.size(3)
        x = F.pad(x, [diffX//2, diffX-diffX//2, diffY//2, diffY-diffY//2])
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return self.scse(x)

## Early-Level with scSE

In [ ]:
class EarlyLevelSCSEUNet(nn.Module):

    def __init__(self, n_classes = 1, in_channels = None, features = [64, 128, 256, 512], use_scse = True, reduction = 16):
        super(EarlyLevelSCSEUNet, self).__init__()

        self.initial = EncoderBlock(in_channels=in_channels, out_channels=features[0], use_scse=use_scse, reduction=reduction)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder1 = EncoderBlock(in_channels=features[0], out_channels=features[0], use_scse=use_scse, reduction=reduction)
        self.encoder2 = EncoderBlock(in_channels=features[0], out_channels=features[1], use_scse=use_scse, reduction=reduction)
        self.encoder3 = EncoderBlock(in_channels=features[1], out_channels=features[2], use_scse=use_scse, reduction=reduction)
        self.encoder4 = EncoderBlock(in_channels=features[2], out_channels=features[3], use_scse=use_scse, reduction=reduction)

        self.center = EncoderBlock(in_channels=features[3], out_channels=features[3], use_scse=use_scse, reduction=reduction)

        self.decoder4 = DecoderBlock(in_channels=features[3], out_channels=features[2], skip_channels=features[3], use_scse=use_scse, reduction=reduction)
        self.decoder3 = DecoderBlock(in_channels=features[2], out_channels=features[1], skip_channels=features[2], use_scse=use_scse, reduction=reduction)
        self.decoder2 = DecoderBlock(in_channels=features[1], out_channels=features[0], skip_channels=features[1], use_scse=use_scse, reduction=reduction)
        self.decoder1 = DecoderBlock(in_channels=features[0], out_channels=features[0], skip_channels=features[0], use_scse=use_scse, reduction=reduction)

        self.up_final = nn.ConvTranspose2d(features[0], features[0], kernel_size=2, stride=2, bias=False)
        self.final = nn.Sequential(nn.Conv2d(features[0], features[0] // 2, kernel_size=3, padding=1, bias=False),
                                   nn.BatchNorm2d(features[0] // 2),
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(features[0] // 2, n_classes, kernel_size=1))

    def forward(self, x):
        e0 = self.initial(x)
        e1 = self.encoder1(self.pool(e0))
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))

        c = self.center(self.pool(e4))

        d4 = self.decoder4(c, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        up = self.up_final(d1)
        return self.final(up)

## Mid-Level with scSE

In [ ]:
class FeatureLevelSCSEUNet(nn.Module):
    def __init__(self, n_classes=1, in_ch_s1=3, in_ch_s2=4, features=[64, 128, 256, 512], reduction=16):
        super(FeatureLevelSCSEUNet, self).__init__()

        self.initial_s1  = EncoderBlock(in_channels=in_ch_s1, out_channels=features[0], use_scse=True, reduction=reduction)
        self.encoder1_s1 = EncoderBlock(in_channels=features[0], out_channels=features[0], use_scse=True, reduction=reduction)
        self.encoder2_s1 = EncoderBlock(in_channels=features[0], out_channels=features[1], use_scse=True, reduction=reduction)
        self.encoder3_s1 = EncoderBlock(in_channels=features[1], out_channels=features[2], use_scse=True, reduction=reduction)
        self.encoder4_s1 = EncoderBlock(in_channels=features[2], out_channels=features[3], use_scse=True, reduction=reduction)

        self.initial_s2 = EncoderBlock(in_channels=in_ch_s2, out_channels=features[0], use_scse=True, reduction=reduction)
        self.encoder1_s2 = EncoderBlock(in_channels=features[0], out_channels=features[0], use_scse=True, reduction=reduction)
        self.encoder2_s2 = EncoderBlock(in_channels=features[0], out_channels=features[1], use_scse=True, reduction=reduction)
        self.encoder3_s2 = EncoderBlock(in_channels=features[1], out_channels=features[2], use_scse=True, reduction=reduction)
        self.encoder4_s2 = EncoderBlock(in_channels=features[2], out_channels=features[3], use_scse=True, reduction=reduction)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.center = nn.Sequential(nn.Conv2d(features[3] * 2, features[3], kernel_size=3, padding=1, bias=False),
                                    nn.BatchNorm2d(features[3]),
                                    nn.ReLU(inplace=True),
                                    nn.Conv2d(features[3], features[3], kernel_size=3, padding=1, bias=False),
                                    nn.BatchNorm2d(features[3]),
                                    nn.ReLU(inplace=True))

        self.center_scse = SCSEBlock(features[3], reduction)

        self.decoder4 = DecoderBlock(in_channels=features[3], out_channels=features[2], skip_channels=features[2] * 2, use_scse=True, reduction=reduction)
        self.decoder3 = DecoderBlock(in_channels=features[2], out_channels=features[1], skip_channels=features[1] * 2, use_scse=True, reduction=reduction)
        self.decoder2 = DecoderBlock(in_channels=features[1], out_channels=features[0], skip_channels=features[0] * 2, use_scse=True, reduction=reduction)
        self.decoder1 = DecoderBlock(in_channels=features[0], out_channels=features[0], skip_channels=features[0] * 2, use_scse=True, reduction=reduction)

        self.final = nn.Sequential(nn.Conv2d(features[0], features[0] // 2, kernel_size=3, padding=1, bias=False),
                                   nn.BatchNorm2d(features[0] // 2),
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(features[0] // 2, n_classes, kernel_size=1))

    def forward(self, x1, x2):
        e0_s1 = self.initial_s1(x1)
        e1_s1 = self.encoder1_s1(self.pool(e0_s1))
        e2_s1 = self.encoder2_s1(self.pool(e1_s1))
        e3_s1 = self.encoder3_s1(self.pool(e2_s1))
        e4_s1 = self.encoder4_s1(self.pool(e3_s1))

        e0_s2 = self.initial_s2(x2)
        e1_s2 = self.encoder1_s2(self.pool(e0_s2))
        e2_s2 = self.encoder2_s2(self.pool(e1_s2))
        e3_s2 = self.encoder3_s2(self.pool(e2_s2))
        e4_s2 = self.encoder4_s2(self.pool(e3_s2))

        e0 = torch.cat([e0_s1, e0_s2], dim=1)
        e1 = torch.cat([e1_s1, e1_s2], dim=1)
        e2 = torch.cat([e2_s1, e2_s2], dim=1)
        e3 = torch.cat([e3_s1, e3_s2], dim=1)
        e4 = torch.cat([e4_s1, e4_s2], dim=1)

        c = self.center(e4)
        c = self.center_scse(c)

        d4 = self.decoder4(c, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        return self.final(d1)

## Late-Level with scSE

In [ ]:
class LateLevelSCSEUNet(nn.Module):
    def __init__(self, n_classes=1, in_ch_s1=3, in_ch_s2=4, features=[64, 128, 256, 512], reduction=16):
        super(LateLevelSCSEUNet, self).__init__()

        self.enc1_0 = EncoderBlock(in_ch_s1, features[0], use_scse=True, reduction=reduction)
        self.enc1_1 = EncoderBlock(features[0], features[1], use_scse=True, reduction=reduction)
        self.enc1_2 = EncoderBlock(features[1], features[2], use_scse=True, reduction=reduction)
        self.enc1_3 = EncoderBlock(features[2], features[3], use_scse=True, reduction=reduction)

        self.dec1_3 = DecoderBlock(features[3], features[2], skip_channels=features[2], use_scse=True, reduction=reduction)
        self.dec1_2 = DecoderBlock(features[2], features[1], skip_channels=features[1], use_scse=True, reduction=reduction)
        self.dec1_1 = DecoderBlock(features[1], features[0], skip_channels=features[0], use_scse=True, reduction=reduction)

        self.enc2_0 = EncoderBlock(in_ch_s2, features[0], use_scse=True, reduction=reduction)
        self.enc2_1 = EncoderBlock(features[0], features[1], use_scse=True, reduction=reduction)
        self.enc2_2 = EncoderBlock(features[1], features[2], use_scse=True, reduction=reduction)
        self.enc2_3 = EncoderBlock(features[2], features[3], use_scse=True, reduction=reduction)

        self.dec2_3 = DecoderBlock(features[3], features[2], skip_channels=features[2], use_scse=True, reduction=reduction)
        self.dec2_2 = DecoderBlock(features[2], features[1], skip_channels=features[1], use_scse=True, reduction=reduction)
        self.dec2_1 = DecoderBlock(features[1], features[0], skip_channels=features[0], use_scse=True, reduction=reduction)

        self.pool = nn.MaxPool2d(2, 2)
        self.final = nn.Sequential(nn.Conv2d(features[0]*2, features[0], kernel_size=3, padding=1, bias=False),
                                   nn.BatchNorm2d(features[0]),
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(features[0], n_classes, kernel_size=1))

    def forward(self, x1, x2):
        e1_0 = self.enc1_0(x1)
        e1_1 = self.enc1_1(self.pool(e1_0))
        e1_2 = self.enc1_2(self.pool(e1_1))
        e1_3 = self.enc1_3(self.pool(e1_2))

        d1_3 = self.dec1_3(e1_3, e1_2)
        d1_2 = self.dec1_2(d1_3, e1_1)
        d1_1 = self.dec1_1(d1_2, e1_0)

        e2_0 = self.enc2_0(x2)
        e2_1 = self.enc2_1(self.pool(e2_0))
        e2_2 = self.enc2_2(self.pool(e2_1))
        e2_3 = self.enc2_3(self.pool(e2_2))

        d2_3 = self.dec2_3(e2_3, e2_2)
        d2_2 = self.dec2_2(d2_3, e2_1)
        d2_1 = self.dec2_1(d2_2, e2_0)

        fused = torch.cat([d1_1, d2_1], dim=1)
        return self.final(fused)

## Single DataLoader

In [ ]:
def make_single_loader(base_dir, split, batch_size, num_classes, augment = False, shuffle = True, generator=None):
    split_root = os.path.join(base_dir, f"{split}_{img_source}")

    imgs_dir  = os.path.join(split_root, "patches", "images")
    masks_dir = os.path.join(split_root, "patches", "masks")

    dataset = DataGeneratorSingle(images_dir=imgs_dir,
                                 masks_dir=masks_dir,
                                 num_classes=num_classes,
                                 augment=augment)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=True, num_workers=0, worker_init_fn=seed_worker, generator=generator)
    return loader

## Dual DataLoader

In [ ]:
def make_dual_loader(base_dir, split, batch_size, num_classes, augment=False, shuffle=True, generator=None):
    s1_root = os.path.join(base_dir, f"{split}_S1")
    s2_root = os.path.join(base_dir, f"{split}_S2")

    imgs1 = os.path.join(s1_root, "patches", "images")
    imgs2 = os.path.join(s2_root, "patches", "images")
    masks = os.path.join(s1_root, "patches", "masks")

    dataset = DataGeneratorDual(images1_dir=imgs1,
                                images2_dir=imgs2,
                                masks_dir=masks,
                                num_classes=num_classes,
                                augment=augment)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=True, num_workers=0, worker_init_fn=seed_worker, generator=generator)
    return loader

## Dual Trainer & Evaluater

In [ ]:
scaler = GradScaler(device.type)
def train_dual(model, dataloader, optimizer, loss_function, device, metrics, scaler):
    model.train()
    total_loss = 0.0
    num_batches = len(dataloader)

    for metric in metrics:
        metric.reset()

    for (x1, x2), labels, _, _ in dataloader:
        x1, x2, labels = x1.to(device), x2.to(device), labels.to(device)
        optimizer.zero_grad()

        output = model(x1, x2)
        logits = output
        loss_output = loss_function(logits, labels)
        loss = loss_output['total'] if isinstance(loss_output, dict) and 'total' in loss_output else loss_output

        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        for metric in metrics:
            metric.update(preds, labels)

        total_loss += loss.item()

    return {"loss": total_loss / num_batches,
            "acc": metrics[0].compute().item(),
            "recall": metrics[1].compute().item(),
            "precision": metrics[2].compute().item(),
            "f1": metrics[3].compute().item()}

def evaluate_dual(model, dataloader, loss_function, device, metrics, threshold=0.5):
    model.eval()
    total_loss = 0.0
    num_batches = len(dataloader)

    for metric in metrics:
        metric.reset()

    with torch.no_grad():
        for (x1, x2), labels, _, _ in dataloader:
            x1, x2, labels = x1.to(device), x2.to(device), labels.to(device)

            logits = model(x1, x2)
            loss_output = loss_function(logits, labels)
            loss = loss_output['total'] if isinstance(loss_output, dict) and 'total' in loss_output else loss_output

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int()

            for metric in metrics:
                metric.update(preds, labels)

            total_loss += loss.item()

    return {"loss": total_loss / num_batches,
            "acc": metrics[0].compute().item(),
            "recall": metrics[1].compute().item(),
            "precision": metrics[2].compute().item(),
            "f1": metrics[3].compute().item()}

## Single Trainer & Evaluater

In [ ]:
scaler = GradScaler(device.type)
def train_single(model, dataloader, optimizer, loss_function, device, metrics, scaler):
    model.train()
    total_loss = 0.0
    num_batches = len(dataloader)

    for metric in metrics:
        metric.reset()

    for batch in dataloader:
        inputs, labels, _, _ = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        output = model(inputs)
        logits = output
        loss_output = loss_function(logits, labels)
        loss = loss_output['total'] if isinstance(loss_output, dict) and 'total' in loss_output else loss_output

        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()

        for metric in metrics:
            metric.update(preds, labels)

        total_loss += loss.item()

    return {"loss": total_loss / num_batches,
            "acc": metrics[0].compute().item(),
            "recall": metrics[1].compute().item(),
            "precision": metrics[2].compute().item(),
            "f1": metrics[3].compute().item()}

def evaluate_single(model, dataloader, loss_function, device, metrics):
    model.eval()
    total_loss  = 0.0
    num_batches = len(dataloader)

    for metric in metrics:
        metric.reset()

    with torch.no_grad():
        for batch in dataloader:
            inputs, labels, _, _ = batch
            inputs, labels = inputs.to(device), labels.to(device)

            output = model(inputs)
            logits = output
            loss_output = loss_function(logits, labels)
            loss = loss_output['total'] if isinstance(loss_output, dict) and 'total' in loss_output else loss_output

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            #preds = remove_singleton_speckles(preds)

            for metric in metrics:
                metric.update(preds, labels) # Removed .int()

            total_loss += loss.item()

    return {"loss": total_loss / num_batches,
            "acc": metrics[0].compute().item(),
            "recall": metrics[1].compute().item(),
            "precision": metrics[2].compute().item(),
            "f1": metrics[3].compute().item()}

## Model Training

In [ ]:
uNetModel = 'Dual' #'Single-Both' #Single-SAR
source_tag   = 'S2_S1' #
source_parts = 'S2', 'S1',  #source_tag.split('_')
num_classes = 1
image_size = 256
num_channels_S1, num_channels_S2 = 3, 4

main_dir = '/content/drive/MyDrive/AI4SF_uNet_Fusion_R1'
data_dir = os.path.join(main_dir,'asia')

images_S1_dir = os.path.join(data_dir,f'image{source_parts[1]}') #
images_S2_dir = os.path.join(data_dir,f'image{source_parts[0]}') #
images_S2S1_dir = os.path.join(data_dir, 'imageS2S1')

masks_dir = os.path.join(data_dir,'masks')
gdf_dir = os.path.join(main_dir,'input','tilesAsia.gpkg')

In [ ]:
if uNetModel == 'Dual':
    train_S2_dir = os.path.join(data_dir, f'train_{source_parts[0]}')
    validate_S2_dir = os.path.join(data_dir, f'validate_{source_parts[0]}')
    test_S2_dir = os.path.join(data_dir, f'test_{source_parts[0]}')

    train_S1_dir = os.path.join(data_dir, f'train_{source_parts[1]}')
    validate_S1_dir = os.path.join(data_dir, f'validate_{source_parts[1]}')
    test_S1_dir = os.path.join(data_dir, f'test_{source_parts[1]}')

elif uNetModel == 'Single-MSI':
    img_source = source_parts[0]
    num_channels = num_channels_S2
    train_S2_dir = os.path.join(data_dir, f'train_{source_parts[0]}')
    validate_S2_dir = os.path.join(data_dir, f'validate_{source_parts[0]}')
    test_S2_dir = os.path.join(data_dir, f'test_{source_parts[0]}')

elif uNetModel == 'Single-SAR':
    img_source = source_parts[1]
    num_channels = num_channels_S1
    train_S1_dir = os.path.join(data_dir, f'train_{source_parts[1]}') #
    validate_S1_dir = os.path.join(data_dir, f'validate_{source_parts[1]}') #
    test_S1_dir = os.path.join(data_dir, f'test_{source_parts[1]}') #

elif uNetModel == 'Single-Both':
    img_source = source_parts[0]+source_parts[1]
    num_channels = num_channels_S1 + num_channels_S2
    train_dir = os.path.join(data_dir, 'train_S2S1')
    validate_dir = os.path.join(data_dir, 'validate_S2S1')
    test_dir = os.path.join(data_dir, 'test_S2S1')

In [ ]:
batch_size = 16
single_models = {'Single-MSI', 'Single-SAR', 'Single-Both'}

if uNetModel in single_models:
    train_dataloader = make_single_loader(data_dir, "train", batch_size, num_classes, augment=False, generator=make_gen(42),  shuffle=True)
    val_dataloader = make_single_loader(data_dir, "validate", batch_size, num_classes, augment=False, generator=make_gen(43),  shuffle=False)
    test_dataloader = make_single_loader(data_dir, "test", batch_size, num_classes, augment=False, generator=make_gen(44),  shuffle=False)
elif uNetModel == 'Dual':
    train_dataloader = make_dual_loader(data_dir, "train", batch_size, num_classes, augment=False, generator=make_gen(42), shuffle=True)
    val_dataloader = make_dual_loader(data_dir, "validate", batch_size, num_classes, augment=False, generator=make_gen(43), shuffle=False)
    test_dataloader = make_dual_loader(data_dir, "test", batch_size, num_classes, augment=False, generator=make_gen(44), shuffle=False)
else:
    raise ValueError(f"Unknown uNetModel: {uNetModel!r}")

In [ ]:
fusion_names = ['Early-scSE'] #Feature-scSE, 'Late-scSE'

num_epochs = 100

pos_weight = compute_pos_weight(train_dataloader, device)
loss_name = 'BCE' #_Tversky_Sobel

criterion = BCETverskySobelLoss(bce_weight=1.0, tversky_weight=0.0, sobel_weight=0.0, tversky_alpha=0.3, tversky_beta=0.7, smooth=1e-6)

loss_function = criterion.to(device)

accuracy_metric = torchmetrics.Accuracy(task="binary").to(device)
recall_metric = torchmetrics.Recall(task="binary").to(device)
precision_metric = torchmetrics.Precision(task="binary").to(device)
f1_metric = torchmetrics.F1Score(task="binary").to(device)

metrics = [accuracy_metric, recall_metric, precision_metric, f1_metric]

best_val_loss = float("inf")


if uNetModel == 'Dual':
    for fusion_name in fusion_names:
        if fusion_name == 'Feature-scSE':
          model = FeatureLevelSCSEUNet(n_classes=num_classes, in_ch_s1=num_channels_S1, in_ch_s2=num_channels_S2)
        elif fusion_name == 'Late-scSE':
          model = LateLevelSCSEUNet(n_classes=num_classes, in_ch_s1=num_channels_S1, in_ch_s2=num_channels_S2)
        model.to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2)
        early_stopping = EarlyStopping(patience=15, loss_name = loss_name, fusion_name = fusion_name)

        for epoch in tqdm(range(num_epochs), desc="Training Progress"):
            train_stats = train_dual(model, train_dataloader, optimizer, loss_function, device, metrics, scaler)
            val_stats = evaluate_dual(model, val_dataloader, loss_function, device, metrics)

            print(f"Epoch {epoch+1}/{num_epochs}")
            print("Train: " + ", ".join([f"{k}={v:.4f}" for k, v in train_stats.items()]))
            print("Val:   " + ", ".join([f"{k}={v:.4f}" for k, v in val_stats.items()]))

            scheduler.step(val_stats['loss'])
            print('Current LR: ', scheduler.get_last_lr())

            torch.cuda.empty_cache()
            gc.collect()

            early_stopping(val_stats['loss'], model, optimizer, scheduler=scheduler, epoch=None)
            if early_stopping.early_stop:
                print("⏹️ Early stopping triggered. Training halted.")
                break

elif uNetModel in single_models:
    model = EarlyLevelSCSEUNet(n_classes = num_classes, in_channels = num_channels)
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    early_stopping = EarlyStopping(patience=10, loss_name = loss_name, fusion_name = uNetModel)

    for epoch in tqdm(range(num_epochs), desc="Training Progress"):
        train_stats = train_single(model, train_dataloader, optimizer, loss_function, device, metrics, scaler)
        val_stats = evaluate_single(model, val_dataloader, loss_function, device, metrics)

        print(f"Epoch {epoch+1}/{num_epochs}")
        print("Train: " + ", ".join([f"{k}={v:.4f}" for k, v in train_stats.items()]))
        print("Val:   " + ", ".join([f"{k}={v:.4f}" for k, v in val_stats.items()]))

        scheduler.step(val_stats['loss'])
        print('Current LR: ', scheduler.get_last_lr())

        torch.cuda.empty_cache()
        gc.collect()

        early_stopping(val_stats['loss'], model, optimizer, scheduler=scheduler, epoch=None)
        if early_stopping.early_stop:
            print("⏹️ Early stopping triggered. Training halted.")
            break

Training Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/100
Train: loss=0.6117, acc=0.7008, recall=0.0867, precision=0.3130, f1=0.1358
Val:   loss=0.6437, acc=0.7217, recall=0.0000, precision=0.0000, f1=0.0000
Current LR:  [0.001]
Epoch 2/100
Train: loss=0.5795, acc=0.7288, recall=0.0000, precision=0.0278, f1=0.0000
Val:   loss=0.5827, acc=0.7217, recall=0.0000, precision=0.0000, f1=0.0000
Current LR:  [0.001]
Epoch 3/100
Train: loss=0.5770, acc=0.7288, recall=0.0000, precision=0.0833, f1=0.0000
Val:   loss=0.5801, acc=0.7217, recall=0.0000, precision=0.0000, f1=0.0000
Current LR:  [0.001]
Epoch 4/100
Train: loss=0.5751, acc=0.7289, recall=0.0000, precision=0.0000, f1=0.0000
Val:   loss=0.5802, acc=0.7217, recall=0.0000, precision=0.0000, f1=0.0000
Current LR:  [0.001]
Epoch 5/100
Train: loss=0.5743, acc=0.7289, recall=0.0000, precision=0.0000, f1=0.0000
Val:   loss=0.5777, acc=0.7217, recall=0.0000, precision=0.0000, f1=0.0000
Current LR:  [0.001]
Epoch 6/100
Train: loss=0.5726, acc=0.7289, recall=0.0000, precision=0.0000, f1=0.000

## Create predictions

In [ ]:
def predict_and_save(model, dataloader, output_dir, threshold=0.5):
    model.eval()
    accuracy_metric.reset()
    recall_metric.reset()
    precision_metric.reset()
    f1_metric.reset()
    os.makedirs(output_dir, exist_ok=True)

    with torch.no_grad():
        for i, (inputs, labels, filenames, extents) in tqdm(
            enumerate(dataloader), desc="Generating Predictions", total=len(dataloader)
        ):
            if isinstance(inputs, (list, tuple)):
                inputs = [inp.to(device) for inp in inputs]
                outputs = model(*inputs)
            else:
                inputs = inputs.to(device)
                outputs = model(inputs)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.sigmoid(outputs)
            preds = (probs >= threshold).int()
            #preds = remove_singleton_speckles(preds)

            accuracy_metric.update(preds, labels.int().to(device))
            precision_metric.update(preds, labels.int().to(device))
            recall_metric.update(preds, labels.int().to(device))
            f1_metric.update(preds, labels.int().to(device))

            for b in range(probs.shape[0]):
                prob_np = probs[b, 0].detach().cpu().numpy().astype(np.float32)
                fname   = filenames[b]

                ds = dataloader.dataset
                candidate_roots = []
                for attr in ["images1_dir", "images2_dir", "images_dir"]:
                    if hasattr(ds, attr) and getattr(ds, attr, None):
                        candidate_roots.append(getattr(ds, attr))

                src_path = None
                for root in candidate_roots:
                    p = os.path.join(root, fname)
                    if os.path.exists(p):
                        src_path = p
                        break
                if src_path is None:
                    raise FileNotFoundError(f"Could not locate source patch for {fname}")

                with rasterio.open(src_path) as src:
                    profile = src.profile
                    profile.update(
                        dtype=rasterio.float32,
                        count=1,
                        compress="DEFLATE",
                        predictor=2,
                        tiled=True,
                        blockxsize=min(256, src.width),
                        blockysize=min(256, src.height)
                    )

                out_path = os.path.join(output_dir, fname)
                with rasterio.open(out_path, 'w', **profile) as dst:
                    dst.write(prob_np, 1)

    accuracy = accuracy_metric.compute().item()
    precision = precision_metric.compute().item()
    recall = recall_metric.compute().item()
    f1 = f1_metric.compute().item()

    print(f"Predictions saved to {output_dir}")

## Create mosaics

In [ ]:
def mosaic_predictions(dataloader, prediction_dir, mosaic_dir, image_size):
    os.makedirs(mosaic_dir, exist_ok=True)
    raster_groups = {}

    for _, (_, _, filenames, _) in tqdm(enumerate(dataloader), desc="Grouping Patches"):
        for fname in filenames:
            parts = fname.split("_")
            if len(parts) < 5:
                print(f"Unexpected filename: {fname}, skipping")
                continue
            try:
                c_off = int(parts[1]); r_off = int(parts[2])
            except ValueError:
                print(f"Offsets not int in {fname}, skipping")
                continue
            key = f"{parts[3]}_{parts[4].split('.')[0]}"
            raster_groups.setdefault(key, []).append((c_off, r_off, fname))

    ds = dataloader.dataset
    candidate_roots = []
    for attr in ["images1_dir", "images2_dir", "images_dir"]:
        if hasattr(ds, attr) and getattr(ds, attr, None):
            candidate_roots.append(getattr(ds, attr))

    if not candidate_roots:
        raise ValueError("Could not determine image root directory from dataloader dataset.")

    tiles_root = os.path.dirname(os.path.dirname(candidate_roots[0]))
    parent_images = os.path.join(tiles_root, "images")

    for key, patches in tqdm(raster_groups.items(), desc="Creating Mosaics"):
        matches = glob.glob(os.path.join(parent_images, f"*{key}.tif"))
        if not matches:
            print(f"No parent for {key}, skipping")
            continue
        parent_path = matches[0]

        with rasterio.open(parent_path) as src:
            transform = src.transform
            crs       = src.crs
            H, W      = src.height, src.width
            acc = np.zeros((H, W), dtype=np.float32)
            cnt = np.zeros((H, W), dtype=np.uint16)

            for c_off, r_off, fname in patches:
                pth = os.path.join(prediction_dir, fname)
                if not os.path.exists(pth):
                    continue
                with rasterio.open(pth) as ps:
                    arr = ps.read(1).astype(np.float32)
                acc[r_off:r_off+image_size, c_off:c_off+image_size] += arr
                cnt[r_off:r_off+image_size, c_off:c_off+image_size] += 1

            mosaic = np.zeros_like(acc, dtype=np.float32)
            np.divide(acc, np.maximum(cnt, 1), out=mosaic, where=(cnt > 0))

        out_fp = os.path.join(mosaic_dir, f"{key}.tif")
        with rasterio.open(
            out_fp, "w", driver="GTiff",
            height=mosaic.shape[0], width=mosaic.shape[1],
            count=1, dtype="float32", crs=crs, transform=transform,
            compress="DEFLATE", predictor=2, tiled=True, blockxsize=512, blockysize=512
        ) as dst:
            dst.write(mosaic, 1)

    print(f"Mosaics saved to {mosaic_dir}")

## Output

In [ ]:
print(f"Evaluating {uNetModel} model...")

if uNetModel in ['Single-MSI', 'Single-SAR', 'Single-Both']:
    if uNetModel == 'Single-MSI':
        model = EarlyLevelSCSEUNet(n_classes=num_classes, in_channels=num_channels_S2)
    elif uNetModel == 'Single-SAR':
        model = EarlyLevelSCSEUNet(n_classes=num_classes, in_channels=num_channels_S1)
    elif uNetModel == 'Single-Both':
        model = EarlyLevelSCSEUNet(n_classes=num_classes, in_channels=num_channels_S1 + num_channels_S2)

    checkpoint_path = os.path.join(base_dir, 'weights', f"{source_tag}_{loss_name}_{uNetModel}_Unet.pt")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)

    predictions_dir = os.path.join('/content/drive/MyDrive/AI4SF_uNet_Fusion_R1/', 'output', f'{source_tag}', f'{loss_name}', uNetModel, 'predictions')
    os.makedirs(predictions_dir, exist_ok=True)

    test_stats = evaluate_single(model, test_dataloader, loss_function, device, metrics)
    print("Test:  " + ", ".join([f"{k}={v:.4f}" for k, v in test_stats.items()]))

    predict_and_save(model, test_dataloader, predictions_dir)
    mosaic_dir = os.path.join('/content/drive/MyDrive/AI4SF_uNet_Fusion_R1/', 'output', f'{source_tag}', f'{loss_name}', uNetModel, 'mosaic')
    mosaic_predictions(test_dataloader, predictions_dir, mosaic_dir, image_size)

elif uNetModel == 'Dual':
    for fusion_name in fusion_names:
        if fusion_name == 'Feature-scSE':
            model = FeatureLevelSCSEUNet(n_classes=num_classes, in_ch_s1=num_channels_S1, in_ch_s2=num_channels_S2)
        elif fusion_name == 'Late-scSE':
            model = LateLevelSCSEUNet(n_classes=num_classes, in_ch_s1=num_channels_S1, in_ch_s2=num_channels_S2)

        checkpoint_path = os.path.join(base_dir, 'weights', f"{source_tag}_{loss_name}_{fusion_name}_Unet.pt")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.to(device)

        predictions_dir = os.path.join('/content/drive/MyDrive/AI4SF_uNet_Fusion_R1/', 'output', f'{source_tag}', f'{loss_name}', fusion_name, 'predictions')
        os.makedirs(predictions_dir, exist_ok=True)

        test_stats = evaluate_dual(model, test_dataloader, loss_function, device, metrics)
        print(f"{fusion_name} Test:  " + ", ".join([f"{k}={v:.4f}" for k, v in test_stats.items()]))

        predict_and_save(model, test_dataloader, predictions_dir)
        #mosaic_dir = os.path.join('/content/drive/MyDrive/AI4SF_uNet_Fusion_R1/', 'output', f'{source_tag}', f'{loss_name}', fusion_name, 'mosaic')
        #mosaic_predictions(test_dataloader, predictions_dir, mosaic_dir, image_size)

Evaluating Single-SAR model...
Test:  loss=0.5065, acc=0.7623, recall=0.2192, precision=0.5291, f1=0.3100


Generating Predictions:   0%|          | 0/8 [00:00<?, ?it/s]

Predictions saved to /content/drive/MyDrive/AI4SF_uNet_Fusion_R1/output/S2_S1/BCE/Single-SAR/predictions


Grouping Patches: 0it [00:00, ?it/s]

Creating Mosaics:   0%|          | 0/10 [00:00<?, ?it/s]

Mosaics saved to /content/drive/MyDrive/AI4SF_uNet_Fusion_R1/output/S2_S1/BCE/Single-SAR/mosaic
